## v1.1 — NaN-native models (XGBoost, HistGradientBoosting)

These models can accept `NaN` directly and learn a split direction for missing
values, so they don't *require* imputation. But per the NaN-handling discussion
in `docs/Nishkarsh/`, two of the four NaN columns are too high-value to leave
to chance:

* `prod_count` — the single strongest driver of `exit_status` (churn rate goes
  1 → 34.5%, 2 → 6.0%, 3 → 88.5%, 4 → 87.5%). Mode-imputing (or trusting a
  generic missing-split) risks systematically mis-imputing the missing 5.4%.
* `acc_balance` — bimodal, and the zero/non-zero split is strongly
  `country`-dependent. A generic missing-split learns this eventually, but a
  contextual median-by-country fill gives it head start.

`country` and `credit_score` are left completely untouched (raw `NaN`,
missingness is MCAR per the earlier EDA, and `credit_score` has ~0 correlation
with everything so there's no context to condition on anyway) — the model's
native missing-value handling deals with those two.

We build **both** a raw-NaN baseline and the smart-imputed version and compare
OOF F1, since it's genuinely an open question whether explicit imputation
still helps a model that can already handle NaN on its own.


In [1]:
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.ensemble import HistGradientBoostingClassifier
from xgboost import XGBClassifier

RANDOM_STATE = 42


In [2]:
train = pd.read_csv('../../data/train.csv')
test = pd.read_csv('../../data/test.csv')
print("train:", train.shape, "test:", test.shape)
print(train.isna().sum())


train: (90000, 14) test: (30000, 13)
id                     0
customer_id            0
last_name              0
credit_score        9556
country             6021
gender                 0
age                    0
tenure                 0
acc_balance         7257
prod_count          4863
has_card               0
is_active              0
estimated_salary       0
exit_status            0
dtype: int64


### Fixed category vocabularies

`country`/`gender`/`prod_count` need to be pandas `category` dtype for both
XGBoost (`enable_categorical=True`) and `HistGradientBoostingClassifier`
(`categorical_features="from_dtype"`) to treat them as categorical rather than
ordinal-numeric. The category *set* has to be identical every time we cast —
train, test, and every CV fold subset — otherwise the two libraries can
silently map the same category string to different internal codes. So we fix
the vocabularies once, up front, from the full training set.


In [3]:
COUNTRY_CATS = sorted(train['country'].dropna().unique().tolist())
GENDER_CATS = sorted(train['gender'].dropna().unique().tolist())
PRODCOUNT_CATS = sorted(train['prod_count'].dropna().unique().astype(int).tolist())

ID_COLS = ['id', 'customer_id', 'last_name']
TARGET = 'exit_status'
PROD_COUNT_IMPUTE_FEATURES = [
    'age', 'is_active', 'acc_balance', 'country', 'credit_score',
    'has_card', 'estimated_salary', 'tenure',
]

print("country:", COUNTRY_CATS)
print("gender:", GENDER_CATS)
print("prod_count:", PRODCOUNT_CATS)


country: ['France', 'Germany', 'Spain']
gender: ['Female', 'Male']
prod_count: [1, 2, 3, 4]


In [4]:
def cast_categoricals(df):
    """Fix dtype only -- never touches which values are missing."""
    df = df.copy()
    df['country'] = df['country'].astype(pd.CategoricalDtype(categories=COUNTRY_CATS))
    df['gender'] = df['gender'].astype(pd.CategoricalDtype(categories=GENDER_CATS))
    if 'prod_count' in df.columns:
        # prod_count may still contain floats (1.0-4.0) or NaN at this point;
        # nullable Int64 lets us round-trip through an integer dtype without
        # NaN blowing up the cast (plain int() astype fails on any NaN present).
        pc_int = df['prod_count'].astype('Int64')
        df['prod_count'] = pc_int.astype(pd.CategoricalDtype(categories=PRODCOUNT_CATS))
    return df


def make_X(df):
    return df.drop(columns=[c for c in ID_COLS + [TARGET] if c in df.columns])


### `BoostingNaNImputer`

Fills only `acc_balance` (median grouped by `country`, `country` NaN treated
as its own group via a sentinel used *only* for the grouping key -- the
`country` column itself is returned untouched) and `prod_count` (predicted by
a small `XGBClassifier` trained on rows with a known `prod_count`, itself fed
raw `NaN`s in `country`/`credit_score` natively -- no leakage into `country`
or `credit_score` at all). Fit only ever sees training-fold data.


In [5]:
class BoostingNaNImputer(BaseEstimator, TransformerMixin):
    def __init__(self, prod_count_features=PROD_COUNT_IMPUTE_FEATURES, random_state=RANDOM_STATE):
        self.prod_count_features = prod_count_features
        self.random_state = random_state

    def _country_key(self, df):
        # sentinel used only to give NaN-country rows their own median group;
        # the real `country` column is never modified by this class.
        return df['country'].astype('object').where(df['country'].notna(), '__NA__')

    def fit(self, X, y=None):
        X = X.copy()
        key = self._country_key(X)
        self.balance_median_by_country_ = X.groupby(key)['acc_balance'].median()
        self.balance_global_median_ = X['acc_balance'].median()

        known = X.dropna(subset=['prod_count'])
        Xk = known[self.prod_count_features].copy()
        Xk['country'] = Xk['country'].astype(pd.CategoricalDtype(categories=COUNTRY_CATS))
        yk_raw = known['prod_count'].astype(int)

        # xgboost's sklearn API requires contiguous zero-indexed class labels
        # (a real behavior in this xgboost version, not just style) -- prod_count
        # is {1,2,3,4}, so remap to positional indices and store the mapping to
        # invert at predict time. Built from whatever classes are actually
        # present in this fold's training data, not assumed to be [1,2,3,4].
        self.prod_count_classes_ = np.sort(yk_raw.unique())
        class_to_idx = {c: i for i, c in enumerate(self.prod_count_classes_)}
        yk = yk_raw.map(class_to_idx)

        self.prod_count_model_ = XGBClassifier(
            n_estimators=300, max_depth=4, learning_rate=0.1,
            enable_categorical=True, tree_method='hist',
            random_state=self.random_state, verbosity=0,
        )
        self.prod_count_model_.fit(Xk, yk)
        return self

    def transform(self, X):
        X = X.copy()
        key = self._country_key(X)
        med_map = key.map(self.balance_median_by_country_)
        X['acc_balance'] = X['acc_balance'].fillna(med_map).fillna(self.balance_global_median_)

        missing = X['prod_count'].isna()
        if missing.any():
            Xm = X.loc[missing, self.prod_count_features].copy()
            Xm['country'] = Xm['country'].astype(pd.CategoricalDtype(categories=COUNTRY_CATS))
            idx_preds = self.prod_count_model_.predict(Xm)
            X.loc[missing, 'prod_count'] = self.prod_count_classes_[idx_preds]
        return X


### OOF cross-validation + F1 threshold search

Both models output a churn *probability*; F1 (unlike ROC-AUC, which the
original Kaggle version of this dataset used) needs a hard 0/1 call, so the
default 0.5 cutoff is not assumed to be optimal. We collect out-of-fold
probabilities across all 5 folds, then search one global threshold on the
combined OOF vector -- more stable than averaging five separately-tuned
per-fold thresholds.


In [6]:
def oof_threshold_search(build_model_fn, X, y, use_imputer, n_splits=5, random_state=RANDOM_STATE):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    oof = np.zeros(len(X))

    for tr_idx, va_idx in skf.split(X, y):
        X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
        y_tr = y.iloc[tr_idx]

        if use_imputer:
            imp = BoostingNaNImputer()
            imp.fit(X_tr)
            X_tr, X_va = imp.transform(X_tr), imp.transform(X_va)

        X_tr, X_va = cast_categoricals(X_tr), cast_categoricals(X_va)

        model = build_model_fn()
        sw = compute_sample_weight('balanced', y_tr)
        model.fit(X_tr, y_tr, sample_weight=sw)
        oof[va_idx] = model.predict_proba(X_va)[:, 1]

    thresholds = np.linspace(0.02, 0.98, 97)
    f1s = [f1_score(y, oof > t) for t in thresholds]
    best = int(np.argmax(f1s))
    return oof, thresholds[best], f1s[best]


In [7]:
X = make_X(train)
y = train[TARGET]

def build_xgb():
    return XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        enable_categorical=True, tree_method='hist',
        random_state=RANDOM_STATE, verbosity=0,
    )

_, thr_xgb_raw, f1_xgb_raw = oof_threshold_search(build_xgb, X, y, use_imputer=False)
_, thr_xgb_imp, f1_xgb_imp = oof_threshold_search(build_xgb, X, y, use_imputer=True)

print(f"XGBoost  raw-NaN     : OOF F1={f1_xgb_raw:.4f}  best_threshold={thr_xgb_raw:.2f}")
print(f"XGBoost  smart-impute: OOF F1={f1_xgb_imp:.4f}  best_threshold={thr_xgb_imp:.2f}")


XGBoost  raw-NaN     : OOF F1=0.6567  best_threshold=0.63
XGBoost  smart-impute: OOF F1=0.6546  best_threshold=0.61


In [8]:
def build_hgb():
    return HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.05, max_iter=400,
        categorical_features='from_dtype',
        random_state=RANDOM_STATE,
    )

_, thr_hgb_raw, f1_hgb_raw = oof_threshold_search(build_hgb, X, y, use_imputer=False)
_, thr_hgb_imp, f1_hgb_imp = oof_threshold_search(build_hgb, X, y, use_imputer=True)

print(f"HistGB   raw-NaN     : OOF F1={f1_hgb_raw:.4f}  best_threshold={thr_hgb_raw:.2f}")
print(f"HistGB   smart-impute: OOF F1={f1_hgb_imp:.4f}  best_threshold={thr_hgb_imp:.2f}")


HistGB   raw-NaN     : OOF F1=0.6562  best_threshold=0.63
HistGB   smart-impute: OOF F1=0.6560  best_threshold=0.63


In [9]:
results = pd.DataFrame([
    {'model': 'XGBoost', 'variant': 'raw-NaN', 'oof_f1': f1_xgb_raw, 'threshold': thr_xgb_raw},
    {'model': 'XGBoost', 'variant': 'smart-impute', 'oof_f1': f1_xgb_imp, 'threshold': thr_xgb_imp},
    {'model': 'HistGB', 'variant': 'raw-NaN', 'oof_f1': f1_hgb_raw, 'threshold': thr_hgb_raw},
    {'model': 'HistGB', 'variant': 'smart-impute', 'oof_f1': f1_hgb_imp, 'threshold': thr_hgb_imp},
]).sort_values('oof_f1', ascending=False).reset_index(drop=True)
results


,model,variant,oof_f1,threshold
0,XGBoost,raw-NaN,0.656714,0.63
1,HistGB,raw-NaN,0.656168,0.63
2,HistGB,smart-impute,0.656044,0.63
3,XGBoost,smart-impute,0.654610,0.61


### Final fit + submission

Refits whichever `(model, variant)` combination scored best in `results`
above on the *full* training set, predicts on `test.csv`, and writes a
submission -- no hardcoded assumption about which of raw-NaN vs
smart-impute wins, since (per the comparison above) that isn't a given.


In [10]:
import os

BUILDERS = {'XGBoost': build_xgb, 'HistGB': build_hgb}
best_row = results.iloc[0]
best_model_name = best_row['model']
best_use_imputer = best_row['variant'] == 'smart-impute'
best_threshold = best_row['threshold']
print(f"Refitting best combo: {best_model_name} / {best_row['variant']} (OOF F1={best_row['oof_f1']:.4f})")

X_train_final = make_X(train)
y_train_final = train[TARGET]
X_test_final = make_X(test)

if best_use_imputer:
    final_imputer = BoostingNaNImputer()
    final_imputer.fit(X_train_final)
    X_train_final = final_imputer.transform(X_train_final)
    X_test_final = final_imputer.transform(X_test_final)

X_train_final = cast_categoricals(X_train_final)
X_test_final = cast_categoricals(X_test_final)

final_model = BUILDERS[best_model_name]()
sw_final = compute_sample_weight('balanced', y_train_final)
final_model.fit(X_train_final, y_train_final, sample_weight=sw_final)

test_proba = final_model.predict_proba(X_test_final)[:, 1]
test_pred = (test_proba > best_threshold).astype(int)

os.makedirs('outputs', exist_ok=True)
submission = pd.DataFrame({'id': test['id'], 'exit_status': test_pred})
submission.to_csv(f'outputs/v1_1_{best_model_name.lower()}_submission.csv', index=False)
submission.head()


Refitting best combo: XGBoost / raw-NaN (OOF F1=0.6567)


,id,exit_status
0,0,0
1,1,1
2,2,1
3,3,0
4,4,0
